# short_volshock_drawdown_drift_reward_short_ppo_screen_v1

Purpose: run the smallest fold-local PPO screen that can falsify the drawdown-drift + trend-preservation reward recipe selected by `short_volshock_drawdown_drift_reward_recipe_screen_v1`. This notebook uses the existing short-volshock de-risk targets, applies the reward patch through `RewardConfig`, and evaluates `wf_2022` plus `wf_2023_2024` against +30pp excess / <=15% MaxDD.

Governance:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer.
- **Direct comparison baselines**: `short_volshock_bc_checkpoint_alignment_v1`, `short_volshock_drawdown_drift_reward_recipe_screen_v1`, and `constrained_checkpoint_selector_v1`.
- **Role**: `diagnose`.
- **Changed layer**: `reward` / short `bc_to_ppo_handoff`; target contract and execution mapping are inherited.
- **Target windows**: `wf_2022` and `wf_2023_2024`; `wf_2020` is skipped because short-volshock targets begin in 2020 and would leak that test year.
- **Accepted trade-off**: PPO may be stopped at the BC initialization if reward shaping worsens held-out MaxDD or relative return.
- **Failure mode**: validation score improves while held-out MaxDD expands, or the candidate fails +30pp relative / <=15% MaxDD on either evaluated fold.

Historical reflection before running:

- Closest success case: `short_volshock_plain` improved `wf_2022` relative return and short-volshock slice accuracy, showing the target family can de-risk stress windows.
- Closest failure case: `short_volshock_weighted_obj025_cb099` improved `wf_2023_2024` MaxDD but still missed relative return and worsened `wf_2022` MaxDD; prior plain short PPO screens also drifted under validation pressure.
- Why this is not a duplicate: the reward patch is now train-consumable in `RewardConfig` and applies drawdown-drift plus trend-preservation pressure during PPO, rather than only screening daily paths or changing BC sample weights.

External basis:

- Qlib workflow separation and replay discipline: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- PPO as bounded fine-tuning: https://arxiv.org/abs/1707.06347
- Constrained policy optimization / hard gate thinking: https://arxiv.org/abs/1705.10528
- FinRL train/validate/test discipline: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Backtest overfitting / multiple-testing discipline: https://doi.org/10.21314/JCF.2016.322


In [1]:
from __future__ import annotations

import copy
import json
import random
import sys
from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from src.soxl_quant.app.constrained_checkpoint_selector import (
    score_constrained_checkpoints,
    summarize_constrained_checkpoint_selection,
)
from src.soxl_quant.app.evaluate import validation_score
from src.soxl_quant.app.selection import train_with_selection
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    summarize_bc_slice_fit,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.environment import build_initial_states
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'short_volshock_drawdown_drift_reward_short_ppo_screen_v1'
CARD = ROOT / 'runtime/reward/card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime/model_selection/card/short_volshock_bc_checkpoint_alignment_v1'
TARGET_CARD = ROOT / 'runtime/model_selection/card/short_volshock_target_generation_handoff_v1'
TARGETS_PATH = TARGET_CARD / 'short_volshock_de_risk_targets.csv'
FRAME_PATH = ROOT / 'data/soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
SHORT_VOLSHOCK_SLICE_ACCURACY_MIN = 0.60
SHORT_VOLSHOCK_SLICE_SAMPLES_MIN = 20
CARD.mkdir(parents=True, exist_ok=True)

WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}


def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def to_json_safe(value):
    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [to_json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(
        json.dumps(to_json_safe(payload), indent=2, sort_keys=True, default=json_default, allow_nan=False) + '\n',
        encoding='utf-8',
    )


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
profile = load_profile(PROFILE_NAME)
REWARD_CARD = ROOT / 'runtime/reward/card/short_volshock_drawdown_drift_reward_recipe_screen_v1'
reward_source_selection = json.loads((REWARD_CARD / 'selection.json').read_text(encoding='utf-8'))
reward_patch = dict(reward_source_selection.get('reward_objective_patch', {}))
profile = replace(profile, reward=replace(profile.reward, **reward_patch))
source_selection = json.loads((SOURCE_CARD / 'selection.json').read_text(encoding='utf-8'))
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])
frame = pd.read_pickle(FRAME_PATH)

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    train_end = pd.Timestamp(spec['train'][1])
    fold_targets = targets[(targets['fold'] == spec['target_fold']) & (targets['datetime'] <= train_end)].copy()
    coverage_rows.append({
        'fold': fold_name,
        'target_fold': spec['target_fold'],
        'target_rows': int(len(fold_targets)),
        'short_volshock_target_rows': int(fold_targets.get('short_volshock_target_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'target_start': str(fold_targets['datetime'].min().date()) if len(fold_targets) else '',
        'target_end': str(fold_targets['datetime'].max().date()) if len(fold_targets) else '',
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'valid_start': spec['valid'][0],
        'valid_end': spec['valid'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
    })
coverage = {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'source_experiment': 'short_volshock_bc_checkpoint_alignment_v1',
    'source_decision': source_selection.get('decision'),
    'source_training_allowed_next': bool(source_selection.get('ppo_screen_allowed_next', source_selection.get('training_allowed_next', False))),
    'reward_recipe_decision': reward_source_selection.get('decision'),
    'reward_training_allowed_next': bool(reward_source_selection.get('training_allowed_next', False)),
    'reward_patch': reward_patch,
    'targets_path': str(TARGETS_PATH.relative_to(ROOT)),
    'feature_frame_path': str(FRAME_PATH.relative_to(ROOT)),
    'windows': coverage_rows,
    'skipped_windows': [
        {
            'fold': 'wf_2020',
            'reason': 'short-volshock targets begin in 2020, so a wf_2020 warm-start would train on the test year',
        }
    ],
}
pd.DataFrame(coverage_rows).to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', coverage)
coverage


{'experiment': 'short_volshock_drawdown_drift_reward_short_ppo_screen_v1',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'source_experiment': 'short_volshock_bc_checkpoint_alignment_v1',
 'source_decision': 'short_volshock_bc_replay_gate_failed_no_promotion',
 'source_training_allowed_next': False,
 'reward_recipe_decision': 'drawdown_drift_reward_recipe_ready_for_short_training_screen',
 'reward_training_allowed_next': True,
 'reward_patch': {'drawdown_drift_penalty': 1.5,
  'drawdown_drift_position_threshold': 0.05,
  'drawdown_drift_reward_enabled': True,
  'drawdown_drift_threshold_pct': 15.0,
  'trend_preservation_floor_enabled': True,
  'trend_preservation_floor_penalty': 1.0,
  'trend_preservation_floor_position': 0.25,
  'trend_preservation_lookback': 63,
  'trend_preservation_min_buyhold_return_pct': 5.0},
 'targets_path': 'runtime/model_selection/card/short_volshock_target_generation_handoff_v1/short_volshock_de_risk_targets.csv',
 'feature_frame_path': 'data/soxl_profile_b

In [3]:
def evaluate_actor_path(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str]) -> dict[str, float]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = dict(split_metrics)
    for src, dst in [
        ('return', 'return'),
        ('sharpe', 'sharpe'),
        ('max_dd', 'max_dd'),
        ('bnh_return', 'buy_and_hold_return'),
        ('relative_outperformance', 'relative_return'),
    ]:
        if src in bt:
            out[dst] = float(bt[src])
    out['backtest_return'] = float(bt.get('return', out.get('return', 0.0)))
    out['backtest_max_dd'] = float(bt.get('max_dd', out.get('max_dd', 0.0)))
    out['backtest_relative_return'] = float(bt.get('relative_outperformance', out.get('relative_return', 0.0)))
    return out


def prefixed_metrics(prefix: str, metrics: dict) -> dict[str, float]:
    keys = ['return', 'sharpe', 'max_dd', 'avg_pos', 'stress_avg_pos', 'buy_and_hold_return', 'relative_return']
    return {f'{prefix}_{k}': float(metrics[k]) for k in keys if k in metrics}


In [4]:
init_qlib(QLIB_DATA_PATH)
train_config = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)
obs_dim = observation_dim_for_profile(profile)

fold_rows = []
history_rows = []
bc_slice_frames = []

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    print(f'\n[fold:{fold_name}] target_fold={spec["target_fold"]} train={spec["train"]} valid={spec["valid"]} test={spec["test"]}')
    set_seed(SEED)
    train_end = pd.Timestamp(spec['train'][1])
    fold_targets = targets[(targets['fold'] == spec['target_fold']) & (targets['datetime'] <= train_end)].copy()
    if fold_targets.empty:
        fold_rows.append({'fold': fold_name, 'status': 'skipped_no_fold_targets'})
        continue

    bc_result = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        train_config=train_config,
        sample_weight_config=None,
        sample_size=0,
    )
    bc_dataset = build_state_objective_bc_dataset_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        sample_size=0,
    )
    bc_slice_metrics = evaluate_bc_actor_slice_fit(
        bc_result.actor,
        bc_result.obs,
        bc_result.actions,
        bc_dataset.metadata,
        slice_cols=('short_volshock_target_state', 'short_volshock_state', 'risk_state', 'is_downtrend_state', 'is_rebound_cluster'),
    )
    bc_slice_summary = summarize_bc_slice_fit(
        bc_slice_metrics,
        required_slices=('short_volshock_target_state',),
        min_accuracy=SHORT_VOLSHOCK_SLICE_ACCURACY_MIN,
        min_samples=SHORT_VOLSHOCK_SLICE_SAMPLES_MIN,
    )
    tagged_slices = bc_slice_metrics.copy()
    tagged_slices.insert(0, 'fold', fold_name)
    bc_slice_frames.append(tagged_slices)

    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    train_df = segment_frame(dataset, 'train')
    valid_df = segment_frame(dataset, 'valid')
    test_df = segment_frame(dataset, 'test')
    price_series = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index().loc[spec['test'][0]:spec['test'][1]]

    train_pickle = CARD / f'{fold_name}_train_frame.pkl'
    train_df.to_pickle(train_pickle)
    initial_states = build_initial_states([(train_pickle, train_df, 0)], profile, smoke=False)
    if not initial_states:
        fold_rows.append({'fold': fold_name, 'status': 'skipped_no_initial_states'})
        continue

    policy, actor = build_policy(obs_dim, profile)
    actor.load_state_dict(copy.deepcopy(bc_result.actor.state_dict()))
    actor.eval()
    bc_test_metrics = evaluate_actor_path(actor, test_df, price_series, profile, spec['test'])
    bc_valid_metrics = evaluate_actor_path(actor, valid_df, dataset.handler.price_frame.droplevel('instrument')['price'].sort_index().loc[spec['valid'][0]:spec['valid'][1]], profile, spec['valid'])
    bc_state = copy.deepcopy(actor.state_dict())

    actor.train()
    selection = train_with_selection(
        policy=policy,
        actor=actor,
        initial_states=initial_states,
        obs_dim=obs_dim,
        profile=profile,
        valid_df=valid_df,
        smoke=False,
        test_df=None,
    )
    selected_test_metrics = evaluate_actor_path(actor, test_df, price_series, profile, spec['test'])
    selected_valid_metrics = selection['best_metrics']
    for item in selection.get('history', []):
        row = dict(item)
        row['fold'] = fold_name
        history_rows.append(row)

    # A conservative drift audit: compare the selected PPO checkpoint to the BC initialization on the held-out fold test.
    rel_delta = float(selected_test_metrics.get('relative_return', 0.0) - bc_test_metrics.get('relative_return', 0.0))
    dd_delta = float(selected_test_metrics.get('max_dd', 0.0) - bc_test_metrics.get('max_dd', 0.0))
    selected_gate = bool(
        selected_test_metrics.get('relative_return', -1e9) >= TARGET_RELATIVE_RETURN
        and selected_test_metrics.get('max_dd', 1e9) <= TARGET_MAX_DD
    )
    bc_gate = bool(
        bc_test_metrics.get('relative_return', -1e9) >= TARGET_RELATIVE_RETURN
        and bc_test_metrics.get('max_dd', 1e9) <= TARGET_MAX_DD
    )
    drift_gate = bool(rel_delta >= -5.0 and dd_delta <= 2.0)

    short_volshock_slice = bc_slice_metrics.set_index('slice').loc['short_volshock_target_state'].to_dict()
    fold_rows.append({
        'fold': fold_name,
        'status': 'evaluated',
        'train_rows': int(len(train_df)),
        'valid_rows': int(len(valid_df)),
        'test_rows': int(len(test_df)),
        'initial_states': int(len(initial_states)),
        'target_rows': int(len(fold_targets)),
        'short_volshock_target_rows': int(fold_targets.get('short_volshock_target_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'bc_short_volshock_accuracy': float(short_volshock_slice['accuracy']),
        'bc_short_volshock_samples': int(short_volshock_slice['samples']),
        'bc_slice_gate': bool(bc_slice_summary['slice_fit_gate']),
        'bc_after_accuracy': float(bc_result.metrics['after_accuracy']),
        'bc_after_loss': float(bc_result.metrics['after_loss']),
        'selected_valid_score': float(selection['best_score']),
        'selected_valid_return': float(selected_valid_metrics.get('return', 0.0)),
        'selected_valid_sharpe': float(selected_valid_metrics.get('sharpe', 0.0)),
        'selected_valid_max_dd': float(selected_valid_metrics.get('max_dd', 0.0)),
        'ppo_vs_bc_relative_delta': rel_delta,
        'ppo_vs_bc_max_dd_delta': dd_delta,
        'bc_gate': bc_gate,
        'selected_gate': selected_gate,
        'drift_gate': drift_gate,
        'fold_gate': bool(selected_gate and drift_gate and bc_slice_summary['slice_fit_gate']),
        **prefixed_metrics('bc_test', bc_test_metrics),
        **prefixed_metrics('selected_test', selected_test_metrics),
        **prefixed_metrics('bc_valid', bc_valid_metrics),
    })

fold_results = pd.DataFrame(fold_rows)
history = pd.DataFrame(history_rows)
bc_slice_metrics_all = pd.concat(bc_slice_frames, ignore_index=True) if bc_slice_frames else pd.DataFrame()

fold_results.to_csv(CARD / 'ppo_fold_results.csv', index=False)
history.to_csv(CARD / 'ppo_selection_history.csv', index=False)
bc_slice_metrics_all.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
for train_pickle in CARD.glob('wf_*_train_frame.pkl'):
    train_pickle.unlink(missing_ok=True)

display(fold_results)
display(history)


[70836:MainThread](2026-05-27 00:31:30,184) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[70836:MainThread](2026-05-27 00:31:30,893) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[70836:MainThread](2026-05-27 00:31:30,893) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}



[fold:wf_2022] target_fold=wf_2022_train train=('2010-03-11', '2020-12-31') valid=('2021-01-01', '2021-12-31') test=('2022-01-01', '2022-12-31')


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,fold,status,train_rows,valid_rows,test_rows,initial_states,target_rows,short_volshock_target_rows,bc_short_volshock_accuracy,bc_short_volshock_samples,...,selected_test_stress_avg_pos,selected_test_buy_and_hold_return,selected_test_relative_return,bc_valid_return,bc_valid_sharpe,bc_valid_max_dd,bc_valid_avg_pos,bc_valid_stress_avg_pos,bc_valid_buy_and_hold_return,bc_valid_relative_return
0,wf_2022,evaluated,2470,252,251,21,253,133,1.000000,133,...,0.075291,-76.614257,71.140743,2.919356,0.245616,14.367137,0.228796,0.0875,78.905912,-67.530831
1,wf_2023_2024,evaluated,2973,250,252,23,756,336,0.925373,335,...,0.087852,-39.952710,32.278777,46.524524,1.666603,16.526914,0.244775,0.0000,79.498558,-46.335383


,iter,score,return,sharpe,max_dd,avg_pos,stress_avg_pos,stress_days,buy_and_hold_return,relative_return,fold
0,0,26.456447,11.375081,0.802160,8.015268,0.228796,0.0875,3.0,78.905912,-67.530831,wf_2022
1,4,28.272135,10.848367,0.927627,9.406507,0.198037,0.1000,3.0,78.905912,-68.057546,wf_2022
2,0,65.907561,33.163175,1.708784,11.927431,0.244775,0.0000,0.0,79.498558,-46.335383,wf_2023_2024
3,4,66.398813,32.274415,1.782049,12.638237,0.228968,0.0000,0.0,79.498558,-47.224143,wf_2023_2024


In [5]:
evaluated = fold_results[fold_results['status'].eq('evaluated')].copy() if not fold_results.empty else pd.DataFrame()

constraint_rows = []
if not evaluated.empty:
    for row in evaluated.to_dict(orient='records'):
        for prefix, candidate in [('bc_test', 'bc_initialization'), ('selected_test', 'reward_ppo_selected')]:
            constraint_rows.append({
                'candidate': candidate,
                'window': row['fold'],
                'relative_return': row.get(f'{prefix}_relative_return', float('nan')),
                'max_dd': row.get(f'{prefix}_max_dd', float('nan')),
            })
constraint_metrics = pd.DataFrame(constraint_rows)
constraint_scores = score_constrained_checkpoints(constraint_metrics) if not constraint_metrics.empty else pd.DataFrame()
constraint_summary = summarize_constrained_checkpoint_selection(constraint_scores)
constraint_scores.to_csv(CARD / 'constrained_checkpoint_scores.csv', index=False)

all_fold_gate = bool((not evaluated.empty) and evaluated['fold_gate'].all() and len(evaluated) == len(WALK_FORWARD_WINDOWS))
any_selected_gate = bool((not evaluated.empty) and evaluated['selected_gate'].any())
all_slice_gate = bool((not evaluated.empty) and evaluated['bc_slice_gate'].all())
any_bc_gate = bool((not evaluated.empty) and evaluated['bc_gate'].any())
all_bc_gate = bool((not evaluated.empty) and evaluated['bc_gate'].all() and len(evaluated) == len(WALK_FORWARD_WINDOWS))
any_drift_fail = bool((not evaluated.empty) and (~evaluated['drift_gate'].astype(bool)).any())
if all_fold_gate:
    decision = 'short_volshock_reward_ppo_screen_passed_partial_wf_no_promotion'
    next_mechanism = 'extend_to_full_walkforward_and_fixed_split_before_promotion'
elif any_bc_gate and any_drift_fail:
    decision = 'short_volshock_reward_ppo_drift_failed_no_promotion'
    next_mechanism = 'do_not_launch_longer_ppo; retain_bc_initialization_for_official_replay_and_constrained_selector_review'
elif any_selected_gate and all_slice_gate:
    decision = 'short_volshock_reward_ppo_screen_mixed_no_promotion'
    next_mechanism = 'inspect_failed_fold_slices_before_longer_training'
else:
    decision = 'short_volshock_reward_ppo_screen_failed_no_promotion'
    next_mechanism = 'drop_drawdown_drift_reward_ppo_or_redesign_reward_timing_before_retry'

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'short_volshock_bc_checkpoint_alignment_v1',
    'reward_recipe_baseline': 'short_volshock_drawdown_drift_reward_recipe_screen_v1',
    'reward_patch': reward_patch,
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'changed_layer': ['reward', 'model_selection'],
    'promotion_allowed': False,
    'promotion_blocker': 'This is a 4-iter fold-local PPO screen using the drawdown-drift/trend-preservation reward patch and skips wf_2020 because no non-leaking short-volshock targets exist for that fold.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'folds_evaluated': evaluated['fold'].tolist() if not evaluated.empty else [],
    'fold_gate_pass_count': int(evaluated['fold_gate'].sum()) if not evaluated.empty and 'fold_gate' in evaluated else 0,
    'selected_gate_pass_count': int(evaluated['selected_gate'].sum()) if not evaluated.empty and 'selected_gate' in evaluated else 0,
    'bc_gate_pass_count': int(evaluated['bc_gate'].sum()) if not evaluated.empty and 'bc_gate' in evaluated else 0,
    'bc_slice_gate_pass_count': int(evaluated['bc_slice_gate'].sum()) if not evaluated.empty and 'bc_slice_gate' in evaluated else 0,
    'all_bc_gate': all_bc_gate,
    'any_drift_fail': any_drift_fail,
    'constrained_selector_decision': constraint_summary.get('decision'),
    'constrained_selector_selected_candidate': constraint_summary.get('selected_candidate'),
    'actor_retention': 'no actor bundle is promoted or retained by this notebook; rerun the notebook to regenerate diagnostic states',
    'sidecars': [
        'coverage.csv',
        'coverage.json',
        'bc_slice_metrics.csv',
        'ppo_selection_history.csv',
        'ppo_fold_results.csv',
        'constrained_checkpoint_scores.csv',
        'selection.json',
        'summary.json',
    ],
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://arxiv.org/abs/1904.04912',
        'https://arxiv.org/abs/1708.02002',
        'https://arxiv.org/abs/1511.05952',
        'https://doi.org/10.21314/JCF.2016.322',
        'https://arxiv.org/abs/1705.10528',
    ],
}
summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'selection': selection,
    'coverage': coverage,
    'fold_results': fold_results.to_dict(orient='records'),
    'constrained_selector': {
        'summary': constraint_summary,
        'scores': constraint_scores.to_dict(orient='records') if not constraint_scores.empty else [],
    },
    'conclusion': [],
}
if evaluated.empty:
    summary['conclusion'].append('No folds were evaluated; do not continue PPO from this handoff.')
else:
    for row in evaluated.to_dict(orient='records'):
        summary['conclusion'].append(
            f"{row['fold']}: BC relative={row.get('bc_test_relative_return', float('nan')):.2f}pp, "
            f"BC MaxDD={row.get('bc_test_max_dd', float('nan')):.2f}%; "
            f"selected relative={row.get('selected_test_relative_return', float('nan')):.2f}pp, "
            f"selected MaxDD={row.get('selected_test_max_dd', float('nan')):.2f}%, "
            f"PPO-vs-BC relative delta={row.get('ppo_vs_bc_relative_delta', float('nan')):.2f}pp, "
            f"DD delta={row.get('ppo_vs_bc_max_dd_delta', float('nan')):.2f}pp, "
            f"fold_gate={bool(row.get('fold_gate', False))}."
        )
    if all_bc_gate and any_drift_fail:
        summary['conclusion'].append(
            'The BC initialization clears both evaluated folds, but reward PPO fails the drift guard by expanding wf_2022 MaxDD; do not run a longer PPO from this reward recipe.'
        )
    summary['conclusion'].append(
        'No runtime promotion: this screen is shorter than full training, omits wf_2020 by leakage guard, and must be followed by full fixed-split + walk-forward validation if any fold is retained.'
    )
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
for line in summary['conclusion']:
    display(Markdown(f'- {line}'))
selection


- wf_2022: BC relative=66.73pp, BC MaxDD=15.00%; selected relative=71.14pp, selected MaxDD=22.78%, PPO-vs-BC relative delta=4.41pp, DD delta=7.78pp, fold_gate=False.

- wf_2023_2024: BC relative=31.12pp, BC MaxDD=13.36%; selected relative=32.28pp, selected MaxDD=13.03%, PPO-vs-BC relative delta=1.15pp, DD delta=-0.33pp, fold_gate=True.

- The BC initialization clears both evaluated folds, but reward PPO fails the drift guard by expanding wf_2022 MaxDD; do not run a longer PPO from this reward recipe.

- No runtime promotion: this screen is shorter than full training, omits wf_2020 by leakage guard, and must be followed by full fixed-split + walk-forward validation if any fold is retained.

{'experiment': 'short_volshock_drawdown_drift_reward_short_ppo_screen_v1',
 'created_at': '2026-05-26T16:36:01.357126+00:00',
 'decision': 'short_volshock_reward_ppo_drift_failed_no_promotion',
 'next_mechanism': 'do_not_launch_longer_ppo; retain_bc_initialization_for_official_replay_and_constrained_selector_review',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'short_volshock_bc_checkpoint_alignment_v1',
 'reward_recipe_baseline': 'short_volshock_drawdown_drift_reward_recipe_screen_v1',
 'reward_patch': {'drawdown_drift_penalty': 1.5,
  'drawdown_drift_position_threshold': 0.05,
  'drawdown_drift_reward_enabled': True,
  'drawdown_drift_threshold_pct': 15.0,
  'trend_preservation_floor_enabled': True,
  'trend_preservation_floor_penalty': 1.0,
  'trend_preservation_floor_position': 0.25,
  'trend_preservation_lookback': 63,
  'trend_preservation_min_buyhold_return_pct': 5.0},
 'role': 'diagnose',
 'profile': 'sparse_release_bc_ppo_scre